# Week 43, Day 3: Continuous attractors: head direction and the ring network

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A self-sustaining bump

Complete simulate(): Euler steps of tau du/dt = -u + W f(u) + I with dt = 0.1, adding optional noise of size noise times sqrt(dt). Cue the network at 1 rad for 300 steps, then remove the cue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1996)
N = 100; th = np.linspace(-np.pi, np.pi, N, endpoint=False)
J0, J1 = -2.0, 6.0
D = th[:, None] - th[None, :]
W = (J0 + J1 * np.cos(D)) / N
f = lambda x: np.clip(x, 0, 1)

def simulate(W, I, T=2000, dt=0.1, tau=1.0, u0=None, noise=0.0):
    u = np.zeros(N) if u0 is None else u0.copy(); hist = []
    for t in range(T):
        inp = I(t) if callable(I) else I
        u += ...
        hist.append(f(u).copy())
    return np.array(hist)

def centre(rates):
    return np.angle((rates * np.exp(1j * th)).sum(-1))

cue = lambda t: (0.5 * np.cos(th - 1.0) + 0.1) if t < 300 else 0.1 * np.ones(N)
h = simulate(W, cue)
print(f"bump centre at the end of the cue {centre(h[299]):.3f} rad, 1700 steps later {centre(h[-1]):.3f} rad; active neurons {(h[-1] > 0.01).sum()}")
plt.imshow(h.T, aspect="auto", origin="lower", extent=[0, 2000, -np.pi, np.pi]); plt.axvline(300, color="w", ls=":")
plt.xlabel("step"); plt.ylabel("preferred direction (rad)"); plt.show()

In [ ]:
import numpy as np
def test_persist():
    "the bump stays within 0.05 rad of the cue direction long after the cue is removed"
    assert abs(centre(h[-1]) - 1.0) < 0.05 and h[-1].max() > 0.3
def test_local():
    "the bump is local: fewer than half the neurons are active"
    assert (h[-1] > 0.01).sum() < 50

test_persist()
test_local()
print("All checks passed")

### Exercise: Speed set by asymmetry

For each gamma in 0, 0.5, 1 and 2, build W_rot = (J0 + J1 cos D + gamma sin D) / N, start from a bump at 0 and run 1500 steps with a uniform input of 0.1. Measure the bump's angular speed (radians per unit time) from the unwrapped centre between steps 500 and 1500.

In [ ]:
speeds = {}
u0 = np.maximum(0, np.cos(th))
for gamma in [0.0, 0.5, 1.0, 2.0]:
    W_rot = ...
    hh = simulate(W_rot, 0.1 * np.ones(N), T=1500, u0=u0)
    c = np.unwrap(centre(hh))
    speeds[gamma] = ...
print({g: round(v, 4) for g, v in speeds.items()}, "rad per unit time")

In [ ]:
import numpy as np
def test_still():
    "with no asymmetry, the bump does not move"
    assert abs(speeds[0.0]) < 1e-3
def test_linear():
    "speed is proportional to asymmetry to within 10%"
    s = np.array([speeds[g] / g for g in [0.5, 1.0, 2.0]])
    assert s.std() / s.mean() < 0.1 and s.mean() > 0

test_still()
test_linear()
print("All checks passed")

### Exercise: A ring with imperfect connections

Add random errors of standard deviation 0.3/N to W, then start bumps at ten random angles and run each for 8000 steps with uniform input 0.1. Record start and end angles in the array ends.

In [ ]:
r4 = np.random.default_rng(1996)
W_het = W + 0.3 * r4.normal(size=W.shape) / N
ends = []
for k in range(10):
    s = r4.uniform(-np.pi, np.pi)
    ...
ends = np.array(ends)
for s, e in ends:
    print(f"start {s:+.2f} rad -> end {e:+.2f} rad")

In [ ]:
import numpy as np
def test_moved():
    "with heterogeneous connections, most bumps have moved more than 0.1 rad"
    moved = np.abs(np.angle(np.exp(1j * (ends[:, 1] - ends[:, 0]))))
    assert np.mean(moved > 0.1) >= 0.5
def test_clustered():
    "the bumps gather: at least five end within 0.05 rad of another bump"
    d = np.abs(np.angle(np.exp(1j * (ends[:, 1][:, None] - ends[:, 1][None]))))
    np.fill_diagonal(d, 9)
    assert (d.min(1) < 0.05).sum() >= 5

test_moved()
test_clustered()
print("All checks passed")